1. Load raw table and create row index

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, monotonically_increasing_id, col

df_arb = spark.table("raw_arboviroses")

w = Window.orderBy(monotonically_increasing_id())

df_arb_idx = df_arb.withColumn("row_id", row_number().over(w))

display(df_arb_idx.select("row_id", "_c0").limit(80))

2. Define valid months and month order

In [0]:
from pyspark.sql.functions import (
    col, lit, expr, trim, upper, when
)
from functools import reduce

meses_validos = [
    "JANEIRO", "FEVEREIRO", "MARÇO", "ABRIL", "MAIO", "JUNHO",
    "JULHO", "AGOSTO", "SETEMBRO", "OUTUBRO", "NOVEMBRO", "DEZEMBRO"
]

mapa_ordem_mes = {
    "JANEIRO": 1,
    "FEVEREIRO": 2,
    "MARÇO": 3,
    "ABRIL": 4,
    "MAIO": 5,
    "JUNHO": 6,
    "JULHO": 7,
    "AGOSTO": 8,
    "SETEMBRO": 9,
    "OUTUBRO": 10,
    "NOVEMBRO": 11,
    "DEZEMBRO": 12
}

3. Define column maps for each disease

In [0]:
mapa_dengue = {
    1998: {"casos": "_c1",  "obitos": None},
    1999: {"casos": "_c2",  "obitos": None},
    2000: {"casos": "_c3",  "obitos": None},
    2001: {"casos": "_c4",  "obitos": None},
    2002: {"casos": "_c5",  "obitos": None},
    2003: {"casos": "_c6",  "obitos": None},
    2004: {"casos": "_c7",  "obitos": None},
    2005: {"casos": "_c8",  "obitos": None},
    2006: {"casos": "_c9",  "obitos": None},
    2007: {"casos": "_c10", "obitos": None},
    2008: {"casos": "_c11", "obitos": None},
    2009: {"casos": "_c12", "obitos": "_c13"},
    2010: {"casos": "_c14", "obitos": "_c15"},
    2011: {"casos": "_c16", "obitos": "_c17"},
    2012: {"casos": "_c18", "obitos": "_c19"},
    2013: {"casos": "_c20", "obitos": "_c21"},
    2014: {"casos": "_c22", "obitos": "_c23"},
    2015: {"casos": "_c25", "obitos": "_c26"},
    2016: {"casos": "_c28", "obitos": "_c29"},
    2017: {"casos": "_c31", "obitos": "_c32"},
    2018: {"casos": "_c34", "obitos": "_c35"},
    2019: {"casos": "_c37", "obitos": "_c38"},
    2020: {"casos": "_c40", "obitos": "_c41"},
    2021: {"casos": "_c43", "obitos": "_c44"},
    2022: {"casos": "_c46", "obitos": "_c47"},
    2023: {"casos": "_c49", "obitos": "_c50"},
    2024: {"casos": "_c52", "obitos": "_c53"},
    2025: {"casos": "_c55", "obitos": "_c56"},
    2026: {"casos": "_c58", "obitos": "_c59"}
}

mapa_zika = {
    2016: {"casos": "_c1",  "obitos": "_c2"},
    2017: {"casos": "_c4",  "obitos": "_c5"},
    2018: {"casos": "_c7",  "obitos": "_c8"},
    2019: {"casos": "_c10", "obitos": "_c11"},
    2020: {"casos": "_c13", "obitos": "_c14"},
    2021: {"casos": "_c16", "obitos": "_c17"},
    2022: {"casos": "_c19", "obitos": "_c20"},
    2023: {"casos": "_c22", "obitos": "_c23"},
    2024: {"casos": "_c25", "obitos": "_c26"},
    2025: {"casos": "_c28", "obitos": "_c29"},
    2026: {"casos": "_c31", "obitos": "_c32"}
}

mapa_chikungunya = {
    2016: {"casos": "_c1",  "obitos": "_c2"},
    2017: {"casos": "_c4",  "obitos": "_c5"},
    2018: {"casos": "_c7",  "obitos": "_c8"},
    2019: {"casos": "_c10", "obitos": "_c11"},
    2020: {"casos": "_c13", "obitos": "_c14"},
    2021: {"casos": "_c16", "obitos": "_c17"},
    2022: {"casos": "_c19", "obitos": "_c20"},
    2023: {"casos": "_c22", "obitos": "_c23"},
    2024: {"casos": "_c25", "obitos": "_c26"},
    2025: {"casos": "_c28", "obitos": "_c29"},
    2026: {"casos": "_c31", "obitos": "_c32"}
}

4. Create reusable transformation function

In [0]:
def transformar_doenca(df_base, row_ini, row_fim, nome_doenca, mapa_colunas):
    df_filtrado = (
        df_base
        .filter((col("row_id") >= row_ini) & (col("row_id") <= row_fim))
        .select(
            trim(upper(col("_c0"))).alias("mes_original"),
            *[col(c) for c in df_base.columns if c != "_c0" and c != "row_id"]
        )
    )

    dfs = []

    for ano, cols in mapa_colunas.items():
        casos_col = cols["casos"]
        obitos_col = cols["obitos"]

        df_ano = (
            df_filtrado
            .select(
                col("mes_original").alias("mes"),
                lit(ano).alias("ano"),
                expr(f"try_cast({casos_col} as int)").alias("casos"),
                (
                    expr(f"try_cast({obitos_col} as int)")
                    if obitos_col is not None
                    else lit(0)
                ).alias("obitos"),
                lit(nome_doenca).alias("doenca")
            )
        )

        dfs.append(df_ano)

    df_saida = reduce(lambda a, b: a.unionByName(b), dfs)

    df_saida = (
        df_saida
        .filter(col("mes").isin(meses_validos))
        .withColumn(
            "ordem_mes",
            when(col("mes") == "JANEIRO", 1)
            .when(col("mes") == "FEVEREIRO", 2)
            .when(col("mes") == "MARÇO", 3)
            .when(col("mes") == "ABRIL", 4)
            .when(col("mes") == "MAIO", 5)
            .when(col("mes") == "JUNHO", 6)
            .when(col("mes") == "JULHO", 7)
            .when(col("mes") == "AGOSTO", 8)
            .when(col("mes") == "SETEMBRO", 9)
            .when(col("mes") == "OUTUBRO", 10)
            .when(col("mes") == "NOVEMBRO", 11)
            .when(col("mes") == "DEZEMBRO", 12)
        )
    )

    return df_saida

5. Transform dengue

In [0]:
df_dengue = transformar_doenca(
    df_base=df_arb_idx,
    row_ini=5,
    row_fim=16,
    nome_doenca="dengue",
    mapa_colunas=mapa_dengue
)

display(df_dengue.orderBy("ano", "ordem_mes"))

6. Transform zika

In [0]:
df_zika = transformar_doenca(
    df_base=df_arb_idx,
    row_ini=26,
    row_fim=37,
    nome_doenca="zika",
    mapa_colunas=mapa_zika
)

display(df_zika.orderBy("ano", "ordem_mes"))

7. Transform chikungunya

In [0]:
df_chikungunya = transformar_doenca(
    df_base=df_arb_idx,
    row_ini=45,
    row_fim=56,
    nome_doenca="chikungunya",
    mapa_colunas=mapa_chikungunya
)

display(df_chikungunya.orderBy("ano", "ordem_mes"))

8. Validate transformed tables

In [0]:
display(df_dengue.orderBy("ano", "ordem_mes"))
display(df_zika.orderBy("ano", "ordem_mes"))
display(df_chikungunya.orderBy("ano", "ordem_mes"))

9. Union final of arboviruses fact table

In [0]:
df_arboviroses_final = (
    df_dengue
    .unionByName(df_zika)
    .unionByName(df_chikungunya)
)

display(
    df_arboviroses_final.orderBy("doenca", "ano", "ordem_mes")
)

10. Quality check

In [0]:
display(
    df_arboviroses_final
    .groupBy("doenca", "ano")
    .count()
    .orderBy("doenca", "ano")
)

11. Final save

In [0]:
df_arboviroses_final.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fato_arboviroses")